# LinearSVC & KNN — Demand Classification

**Models:** Linear SVC · K-Nearest Neighbors  
**Target:** `demand_label` (binary: 0 = low demand, 1 = high demand)  
**Tuning:** Optuna (weighted-F1 objective)  

> **⚠️ Scale sensitivity:** Both SVM and KNN are distance-based — the features are already StandardScaled by `postprocessing.py`, which is critical for these models.  
> **⚠️ Runtime:** SVM with `rbf` kernel scales as O(n²–n³). If training is too slow, reduce `n_trials` or use a data subsample for tuning.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.svm       import LinearSVC
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split, learning_curve
from sklearn.metrics import (
    classification_report, confusion_matrix, ConfusionMatrixDisplay,
    accuracy_score, f1_score,
)
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

plt.style.use('dark_background')
SEED = 42

## 1. Load Data

In [2]:
train_df = pd.read_csv('../../data/splits/train.csv', low_memory=False)
test_df  = pd.read_csv('../../data/splits/test.csv',  low_memory=False)

TARGET = 'demand_label'

DROP_COLS = [
    'demand_label', 'demand_label_3', 'demand_score',
    'Price_log', 'Price_original',
    'Price_vs_city_median',
]
raw_cols     = [c for c in train_df.columns if c.endswith('_raw')]
amenity_cols = [c for c in train_df.columns if 'Parsed Amenities' in c]
DROP_COLS   += raw_cols + amenity_cols

feature_cols = [c for c in train_df.columns if c not in DROP_COLS]

X_all  = train_df[feature_cols].fillna(0)
y_all  = train_df[TARGET]
X_test = test_df[feature_cols].fillna(0)
y_test = test_df[TARGET]

# Sanitize column names for LightGBM (remove special JSON characters)
import re
X_all.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in X_all.columns]
X_test.columns = [re.sub(r"[^A-Za-z0-9_]+", "_", c) for c in X_test.columns]

X_train, X_val, y_train, y_val = train_test_split(
    X_all, y_all, test_size=0.2, random_state=SEED, stratify=y_all
)

print(f'Train : {X_train.shape} | Val : {X_val.shape} | Test : {X_test.shape}')

# ── Optional: subsample for faster SVM/KNN tuning on large datasets ──
# Uncomment if training takes too long:
# from sklearn.utils import resample
# X_train_sml, y_train_sml = resample(X_train, y_train, n_samples=50_000, random_state=SEED, stratify=y_train)
X_train_sml, y_train_sml = X_train, y_train  # use full set by default

Train : (311661, 97) | Val : (77916, 97) | Test : (97395, 97)


---
## 2. Linear SVC

In [ ]:
def objective_svm(trial):
    params = {
        'C':      trial.suggest_float('C',     1e-3, 100, log=True),
    }
    model = LinearSVC(**params, class_weight='balanced', random_state=SEED, dual='auto', max_iter=5000)
    model.fit(X_train_sml, y_train_sml)
    return f1_score(y_val, model.predict(X_val), average='weighted')

study_svm = optuna.create_study(direction='maximize')
study_svm.optimize(objective_svm, n_trials=50, n_jobs=2)
print(f'Best Val F1 (LinearSVC) : {study_svm.best_value:.4f}')
print(f'Best Params            : {study_svm.best_params}')

In [ ]:
X_combined = pd.concat([X_train, X_val])
y_combined = pd.concat([y_train, y_val])

svm_model = LinearSVC(**study_svm.best_params, class_weight='balanced', random_state=SEED, dual='auto', max_iter=5000)
svm_model.fit(X_combined, y_combined)
print('LinearSVC trained.')

### LinearSVC — Train & Validation Results

In [ ]:
y_pred_train = svm_model.predict(X_train)

print('=== LinearSVC — Train Set ===')
print(f'Accuracy      : {accuracy_score(y_train, y_pred_train):.4f}')
print(f'F1 (weighted) : {f1_score(y_train, y_pred_train, average="weighted"):.4f}')
print(classification_report(y_train, y_pred_train, target_names=['Low','High']))

y_pred_val = svm_model.predict(X_val)

print('=== LinearSVC — Validation Set ===')
print(f'Accuracy      : {accuracy_score(y_val, y_pred_val):.4f}')
print(f'F1 (weighted) : {f1_score(y_val, y_pred_val, average="weighted"):.4f}')
print(classification_report(y_val, y_pred_val, target_names=['Low','High']))


In [ ]:
y_pred_svm = svm_model.predict(X_test)

print('=== LinearSVC ===')
print(f'Accuracy      : {accuracy_score(y_test, y_pred_svm):.4f}')
print(f'F1 (weighted) : {f1_score(y_test, y_pred_svm, average="weighted"):.4f}')
print(classification_report(y_test, y_pred_svm, target_names=['Low','High']))

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_svm), display_labels=['Low','High']).plot(ax=ax, colorbar=False, cmap='Blues')
ax.set_title('Confusion Matrix — LinearSVC', fontsize=14); plt.tight_layout(); plt.show()

In [ ]:
lc_svm = LinearSVC(**study_svm.best_params, class_weight='balanced', random_state=SEED, dual='auto', max_iter=5000)
ts, tr_s, val_s = learning_curve(
    lc_svm, X_train_sml, y_train_sml,
    cv=5, scoring='f1_weighted',
    train_sizes=np.linspace(0.1,1.0,10), n_jobs=-1,
)
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(ts, tr_s.mean(1), 'o-', label='Train F1'); ax.fill_between(ts, tr_s.mean(1)-tr_s.std(1), tr_s.mean(1)+tr_s.std(1), alpha=0.2)
ax.plot(ts, val_s.mean(1), 's-', label='Val F1');   ax.fill_between(ts, val_s.mean(1)-val_s.std(1), val_s.mean(1)+val_s.std(1), alpha=0.2)
ax.set(xlabel='Training Size', ylabel='F1 (Weighted)', title='Learning Curve — LinearSVC')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 3. K-Nearest Neighbors

In [ ]:
def objective_knn(trial):
    params = {
        'n_neighbors': trial.suggest_int('n_neighbors', 1, 50),
        'weights':     trial.suggest_categorical('weights', ['uniform', 'distance']),
        'metric':      trial.suggest_categorical('metric', ['euclidean', 'manhattan', 'minkowski']),
        'p':           trial.suggest_int('p', 1, 3),   # only used by minkowski
    }
    model = KNeighborsClassifier(**params, n_jobs=-1)
    model.fit(X_train, y_train)
    return f1_score(y_val, model.predict(X_val), average='weighted')

study_knn = optuna.create_study(direction='maximize')
study_knn.optimize(objective_knn, n_trials=300, n_jobs=2)
print(f'Best Val F1 (KNN) : {study_knn.best_value:.4f}')
print(f'Best Params       : {study_knn.best_params}')

In [ ]:
knn_model = KNeighborsClassifier(**study_knn.best_params, n_jobs=-1)
knn_model.fit(X_combined, y_combined)
print('KNN trained.')

### KNN — Train & Validation Results

In [ ]:
# --- Results on Train Set ---
y_pred_train = knn_model.predict(X_train)

print('=== KNN — Train Set ===')
print(f'Accuracy      : {accuracy_score(y_train, y_pred_train):.4f}')
print(f'F1 (weighted) : {f1_score(y_train, y_pred_train, average="weighted"):.4f}')
print(classification_report(y_train, y_pred_train, target_names=['Low','High']))

# --- Results on Validation Set ---
y_pred_val = knn_model.predict(X_val)

print('=== KNN — Validation Set ===')
print(f'Accuracy      : {accuracy_score(y_val, y_pred_val):.4f}')
print(f'F1 (weighted) : {f1_score(y_val, y_pred_val, average="weighted"):.4f}')
print(classification_report(y_val, y_pred_val, target_names=['Low','High']))


In [ ]:
y_pred_knn = knn_model.predict(X_test)

print('=== KNN ===')
print(f'Accuracy      : {accuracy_score(y_test, y_pred_knn):.4f}')
print(f'F1 (weighted) : {f1_score(y_test, y_pred_knn, average="weighted"):.4f}')
print(classification_report(y_test, y_pred_knn, target_names=['Low','High']))

fig, ax = plt.subplots(figsize=(6,5))
ConfusionMatrixDisplay(confusion_matrix(y_test, y_pred_knn), display_labels=['Low','High']).plot(ax=ax, colorbar=False, cmap='Greens')
ax.set_title('Confusion Matrix — KNN', fontsize=14); plt.tight_layout(); plt.show()

In [ ]:
lc_knn = KNeighborsClassifier(**study_knn.best_params, n_jobs=-1)
ts, tr_s, val_s = learning_curve(lc_knn, X_train, y_train, cv=5, scoring='f1_weighted', train_sizes=np.linspace(0.1,1.0,10), n_jobs=-1)
fig, ax = plt.subplots(figsize=(9,5))
ax.plot(ts, tr_s.mean(1), 'o-', label='Train F1'); ax.fill_between(ts, tr_s.mean(1)-tr_s.std(1), tr_s.mean(1)+tr_s.std(1), alpha=0.2)
ax.plot(ts, val_s.mean(1), 's-', label='Val F1');   ax.fill_between(ts, val_s.mean(1)-val_s.std(1), val_s.mean(1)+val_s.std(1), alpha=0.2)
ax.set(xlabel='Training Size', ylabel='F1 (Weighted)', title='Learning Curve — KNN')
ax.legend(); ax.grid(alpha=0.3); plt.tight_layout(); plt.show()

---
## 4. Model Comparison

In [ ]:
results = pd.DataFrame({
    'Model':         ['LinearSVC', 'KNN'],
    'Accuracy':      [accuracy_score(y_test, p) for p in [y_pred_svm, y_pred_knn]],
    'F1 (weighted)': [f1_score(y_test, p, average='weighted') for p in [y_pred_svm, y_pred_knn]],
}).set_index('Model')
print(results.round(4))

results.plot(kind='bar', figsize=(7,5), rot=0, colormap='coolwarm')
plt.title('LinearSVC vs KNN — Test Set Performance')
plt.ylabel('Score'); plt.ylim(0.5, 1.0); plt.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()